# ZeroJudge 5 題完整解題筆記

這份 Notebook 合併 5 題的題意整理、解題想法、Python 程式碼與範例測試。


---


# a181 逆逆向思考：Trie 字典樹

適合：國中初學者／APCS 入門練習

## 題目重點
輸入多組資料。每組第一行是 N，接著有 N 個字串。目標是建立 Trie，並依照 ASCII 由小到大輸出 Trie 中保存的字串。

Trie 可以想成「把有共同前綴的字串共用同一段路」。例如 `tea`、`ted`、`ten` 都會共用 `t -> e` 這段。

## 解題想法
1. 用巢狀 dictionary 表示 Trie。
2. 插入字串時，一個字元一個字元往下走。
3. 字串結束時，在節點放一個特殊記號 `END`。
4. 輸出時做 DFS，並且每一層的 key 用 `sorted()` 排序，Python 對字元排序預設就是依 Unicode/ASCII 順序，對本題可用。
5. 題目可能有很多組測資，所以要一直讀到 EOF。

## 可以提交到 ZeroJudge 的完整程式
下面這格是完整解答。提交時只需要複製 code cell 的內容。

In [ ]:
import sys

END = ""   # 特殊記號：代表這裡是一個完整字串的結尾


def insert(root, word):
    """把一個字串放進 Trie。"""
    node = root
    for ch in word:
        if ch not in node:
            node[ch] = {}
        node = node[ch]
    node[END] = {}   # 記錄：這個位置是一個完整字串


def collect(node, path, answer):
    """依照 ASCII 順序，把 Trie 裡面的完整字串收集出來。"""
    if END in node:
        answer.append(path)

    for ch in sorted(node.keys()):
        if ch == END:
            continue
        collect(node[ch], path + ch, answer)


def solve(data):
    lines = data.splitlines()
    i = 0
    output = []

    while i < len(lines):
        if lines[i].strip() == "":
            i += 1
            continue

        n = int(lines[i])
        i += 1

        root = {}
        for _ in range(n):
            word = lines[i].rstrip("\n")
            i += 1
            insert(root, word)

        answer = []
        collect(root, "", answer)
        output.extend(answer)

    return "\n".join(output)


if __name__ == "__main__":
    print(solve(sys.stdin.read()))

## 範例測試／觀察

In [ ]:
sample = """8
A
to
tea
ted
ten
i
in
inn
"""
print(solve(sample))


---


# f581 圓環出口：前綴和 + 二分搜尋

適合：國中初學者／APCS 入門練習

## 題目重點
有 n 個房間排成一圈，從 i 會走到 `(i+1) mod n`。進入房間 i 可以拿到 `p[i]` 分。每個任務需要累積 `q` 分，達成任務後會停在「最後拿分房間的下一個房間」。最後輸出所有任務做完後的位置。

## 解題想法
1. 因為房間是圓形，直接把陣列複製兩次：`p + p`。
2. 建立前綴和 `prefix`，這樣可以快速知道一段連續房間的總分。
3. 每次從目前位置 `pos` 開始，要找最小的 `end`，使得從 `pos` 到 `end` 的分數總和 >= q。
4. 這可以用 `bisect_left` 在前綴和上做二分搜尋。
5. 因為題目保證每個 q 不會超過全部房間總和，所以最多走一圈就一定完成。

## 可以提交到 ZeroJudge 的完整程式
下面這格是完整解答。提交時只需要複製 code cell 的內容。

In [ ]:
import sys
from bisect import bisect_left


def solve(data):
    nums = list(map(int, data.split()))
    idx = 0
    n = nums[idx]
    m = nums[idx + 1]
    idx += 2

    p = nums[idx:idx + n]
    idx += n
    qs = nums[idx:idx + m]

    double_p = p + p

    prefix = [0]
    for x in double_p:
        prefix.append(prefix[-1] + x)

    pos = 0
    for q in qs:
        start_sum = prefix[pos]
        target = start_sum + q

        # 找到第一個 prefix[k] >= target
        # k 代表已經走過的下一格位置，所以最後拿分房間是 k-1
        k = bisect_left(prefix, target, pos + 1, pos + n + 1)

        # 完成任務後停在最後拿分房間的下一格
        pos = k % n

    return str(pos)


if __name__ == "__main__":
    print(solve(sys.stdin.read()))

## 範例測試／觀察

In [ ]:
print(solve("""7 3
2 1 5 4 3 5 3
8 9 12
"""))  # 4
print(solve("""4 3
1 3 5 7
4 2 2
"""))  # 0


---


# e313 最少相異字母：set + 自訂排序

適合：國中初學者／APCS 入門練習

## 題目重點
給 N 個只含大寫 A~Z 的字串。要找出「相異字母數量最少」的字串。如果有多個一樣少，選字典序最小的字串。

## 解題想法
1. `set(s)` 可以把重複字母去掉，所以 `len(set(s))` 就是相異字母數量。
2. 每個字串都可以變成一個比較用的 key：`(相異字母數量, 字串本身)`。
3. Python 比 tuple 時會先比第一項；第一項一樣才比第二項。
4. 因此 `min(words, key=lambda s: (len(set(s)), s))` 就剛好符合題意。

## 可以提交到 ZeroJudge 的完整程式
下面這格是完整解答。提交時只需要複製 code cell 的內容。

In [ ]:
import sys


def solve(data):
    lines = data.splitlines()
    n = int(lines[0])
    words = [lines[i].strip() for i in range(1, n + 1)]

    best = min(words, key=lambda s: (len(set(s)), s))
    return best


if __name__ == "__main__":
    print(solve(sys.stdin.read()))

## 範例測試／觀察

In [ ]:
print(solve("""3
ABBCAAB
AABBACC
AAPPCCSS
"""))  # AABBACC


---


# q368 阿克曼函數：看懂遞迴後化成公式

適合：國中初學者／APCS 入門練習

## 題目重點
輸入 `m, n`，範圍是 `0 <= m <= 3`、`0 <= n <= 11`，輸出 Ackermann function `A(m, n)`。

定義：
- `A(0, n) = n + 1`
- `A(m, 0) = A(m-1, 1)`，當 `m > 0`
- `A(m, n) = A(m-1, A(m, n-1))`，當 `m > 0` 且 `n > 0`

## 解題想法
直接照遞迴寫也可以，但阿克曼函數長很快。這題 m 最大只有 3，所以可以先推成公式：

- `A(0, n) = n + 1`
- `A(1, n) = n + 2`
- `A(2, n) = 2n + 3`
- `A(3, n) = 2^(n+3) - 3`

這樣不需要真的做很多層遞迴，速度快，也不會卡住。

## 可以提交到 ZeroJudge 的完整程式
下面這格是完整解答。提交時只需要複製 code cell 的內容。

In [ ]:
import sys


def ackermann(m, n):
    if m == 0:
        return n + 1
    if m == 1:
        return n + 2
    if m == 2:
        return 2 * n + 3
    # m == 3
    return 2 ** (n + 3) - 3


def solve(data):
    m, n = map(int, data.split())
    return str(ackermann(m, n))


if __name__ == "__main__":
    print(solve(sys.stdin.read()))

## 範例測試／觀察

In [ ]:
print(solve("1 2"))   # 4
print(solve("3 11"))  # 16381


---


# g277 幸運數字：笛卡爾樹 + 子樹總和

適合：國中初學者／APCS 入門練習

## 題目重點
在一段區間中，先找最小值的位置。用這個位置把區間切成左邊和右邊，比較兩邊總和：
- 左邊總和比較大：幸運數字在左邊
- 右邊總和比較大或相等：幸運數字在右邊
一直重複，直到剩下一個數字，輸出它。

## 解題想法
暴力每次找區間最小值會太慢。比較好的方法是建立「最小值笛卡爾樹」：

1. 陣列中最小的數會是整棵樹的根。
2. 最小值左邊的區間會形成左子樹；右邊的區間會形成右子樹。
3. 題目的每一步，其實就是在這棵樹上往左子樹或右子樹走。
4. 只要先算好每個節點的子樹總和，就可以比較左、右總和。
5. 由於所有號碼都不同，建樹時不會有相同大小的問題。

## 可以提交到 ZeroJudge 的完整程式
下面這格是完整解答。提交時只需要複製 code cell 的內容。

In [ ]:
import sys


def build_min_cartesian_tree(a):
    """建立最小值笛卡爾樹，回傳 root, left, right。"""
    n = len(a)
    left = [-1] * n
    right = [-1] * n
    parent = [-1] * n
    stack = []

    for i, x in enumerate(a):
        last = -1

        # 維持 stack 裡面的值是遞增的
        while stack and a[stack[-1]] > x:
            last = stack.pop()

        if stack:
            right[stack[-1]] = i
            parent[i] = stack[-1]

        if last != -1:
            left[i] = last
            parent[last] = i

        stack.append(i)

    root = 0
    while parent[root] != -1:
        root = parent[root]

    return root, left, right


def solve(data):
    nums = list(map(int, data.split()))
    n = nums[0]
    a = nums[1:1 + n]

    root, left, right = build_min_cartesian_tree(a)

    # 用後序順序計算每個節點的子樹總和
    subtree_sum = [0] * n
    order = [root]
    for node in order:
        if left[node] != -1:
            order.append(left[node])
        if right[node] != -1:
            order.append(right[node])

    for node in reversed(order):
        total = a[node]
        if left[node] != -1:
            total += subtree_sum[left[node]]
        if right[node] != -1:
            total += subtree_sum[right[node]]
        subtree_sum[node] = total

    # 從根開始，依題意選左邊或右邊
    cur = root
    while left[cur] != -1 or right[cur] != -1:
        left_sum = subtree_sum[left[cur]] if left[cur] != -1 else 0
        right_sum = subtree_sum[right[cur]] if right[cur] != -1 else 0

        if left_sum > right_sum:
            cur = left[cur]
        else:
            cur = right[cur]

    return str(a[cur])


if __name__ == "__main__":
    print(solve(sys.stdin.read()))

## 範例測試／觀察

In [ ]:
print(solve("""5
4 2 3 1 5
"""))  # 4
print(solve("""8
3 9 4 5 1 6 2 8
"""))  # 9